In [ ]:
import rdkit.Chem as Chem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')


In [ ]:
# Vortioxetine as reference (loaded from the uploaded SDF)
from rdkit.Chem import SDMolSupplier, rdDistGeom, rdShapeAlign, AddHs

refmol = next(m for m in SDMolSupplier("data/vortioxetine.sdf") if m is not None)
print(f"Reference: {Chem.MolToSmiles(refmol)}")

def get_alignment_scores(mols, ref=refmol):
    scores = []
    poses = []
    for m in mols:
        try:
            m = AddHs(m)
            rdDistGeom.EmbedMultipleConfs(m, numConfs=20, randomSeed=0xf00d,numThreads=-1)
            conf_scores = []
            for confId in range(m.GetNumConformers()):
                conf_scores.append(
                    sum(rdShapeAlign.AlignMol(ref, m, probeConfId=confId)) / 2
                )
            scores.append(max(conf_scores) if conf_scores else 0.0)
        except Exception:
            scores.append(0.0)
    return scores


In [ ]:
get_alignment_scores([Chem.MolFromSmiles("Cc1ccc(Sc2ccccc2N2CCNCC2)c(C)c1")])

In [ ]:
from lacan import lacan,gen
from rdkit.Chem import Draw
from lacan.gen import GAReporter
p = lacan.load_profile("chembl")

reporter = GAReporter(label="Penalized LogP")
winners_shape = gen.generate_optimized_molecules(
    get_alignment_scores, p,
    preset="medium",
    # scoring_budget=20,    # molecules scored per generation (preset default)
    # startN=20,            # initial population size
    generations=15,
    higher_is_better=True,
    n_jobs=12,
    quiet=False,
    seed=222,
    callback=reporter
)
allc_shape = sorted(winners_shape, key=lambda x: -x[1])
print(f"\n{len(allc_shape)} molecules in HallOfFame")
d = Draw.MolsToGridImage(
    [Chem.MolFromSmiles(c[0]) for c in allc_shape],
    useSVG=True, molsPerRow=4,
    legends=[str(round(c[1], 3)) for c in allc_shape],
    maxMols=20,
)
display(d)

In [ ]:
allc_shape = sorted(winners_shape, key=lambda x: -x[1])
print(f"\n{len(allc_shape)} molecules in HallOfFame")
d = Draw.MolsToGridImage(
    [Chem.MolFromSmiles(c[0]) for c in allc_shape],
    useSVG=True, molsPerRow=4,
    legends=[str(round(c[1], 3)) for c in allc_shape],
    maxMols=8,
)
display(d)

In [ ]:
penalized_logp(Chem.MolFromSmiles("c1ccccc1CCCCCCc1ccccc1CCCCCCCCCCCCCCCCCc1ccccc1"))

In [ ]:
reporter.label = "ComboTanimoto to vortioxetine"
reporter.plot()

In [ ]:
import py3Dmol
from rdkit import Chem
from rdkit.Chem import MolToMolBlock, AddHs, SDMolSupplier
from rdkit.Chem import rdDistGeom, rdShapeAlign

# ── Load reference ────────────────────────────────────────────────────────────
refmol = next(m for m in SDMolSupplier("data/vortioxetine.sdf") if m is not None)

# ── Colour palette for top winners ───────────────────────────────────────────
WINNER_COLOURS = [
    'cyanCarbon', 'greenCarbon', '#45B7D1', '#96CEB4', '#FFEAA7',
    '#DDA0DD', '#98D8C8', '#F7DC6F', '#BB8FCE', '#82E0AA',
]

def best_aligned_molblock(mol_2d, ref=refmol, n_confs=20, seed=0xf00d):
    """Embed mol_2d, align all confs to ref, return (molblock, score) of best conf."""
    m = AddHs(mol_2d)
    rdDistGeom.EmbedMultipleConfs(m, numConfs=n_confs, randomSeed=seed, numThreads=-1)
    if m.GetNumConformers() == 0:
        return None, 0.0
    best_sc, best_cid = -1, 0
    for cid in range(m.GetNumConformers()):
        sc = sum(rdShapeAlign.AlignMol(ref, m, probeConfId=cid)) / 2
        if sc > best_sc:
            best_sc, best_cid = sc, cid
    rw = Chem.RWMol(m)
    for c in list(rw.GetConformers()):
        if c.GetId() != best_cid:
            rw.RemoveConformer(c.GetId())
    return MolToMolBlock(rw), best_sc

# ── Pick top N winners from GA ────────────────────────────────────────────────
TOP_N = 1  # adjust as needed
top_winners = allc_shape[:TOP_N]  # already sorted by score descending

print(f"Overlaying top {len(top_winners)} GA winner(s) + reference")
print(f"{'Rank':<6} {'Score':<8} {'SMILES'}")
print("-" * 80)

# ── Build py3Dmol viewer ──────────────────────────────────────────────────────
view = py3Dmol.view(width=800, height=600)

# Reference — white sticks (model 0)
view.addModel(MolToMolBlock(refmol), 'mol')
view.setStyle({'model': 0}, {'stick': {'colorscheme': 'purpleCarbon', 'radius': 0.15}})
view.addSurface(py3Dmol.VDW, {'opacity': 0.08, 'color': 'white'}, {'model': 0})

# GA winners — colour-coded by rank
model_idx = 1
for rank, (smi, ga_score) in enumerate(top_winners):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print(f"  [{rank+1}] Could not parse SMILES, skipping: {smi}")
        continue

    molblock, align_score = best_aligned_molblock(mol)
    if molblock is None:
        print(f"  [{rank+1}] Embedding failed, skipping: {smi}")
        continue

    colour = WINNER_COLOURS[rank % len(WINNER_COLOURS)]
    print(f"  {rank+1:<4} GA={ga_score:.3f}  align={align_score:.3f}  colour={colour}  {smi}")

    view.addModel(molblock, 'mol')
    view.setStyle({'model': model_idx}, {'stick': {'colorscheme': colour, 'radius': 0.12}})
    model_idx += 1

print(f"\nShowing {model_idx - 1} winner(s) + reference (white)")
view.zoomTo()
view.show()

In [ ]:
wri = Chem.SDWriter("D2_docking_poses.sdf")
for a in [aux,aux2,aux3,aux4]:
    wri.write(a["ligand"])
wri.close()